# Barcode outlier-filtering frequency

How often the extreme-RNA-barcode (hyperactive-integration) filtering removes barcodes: on average 3.58% of barcodes were removed per replicate (324,822 in total; 4.39 per cCRE).

In [ ]:
import pandas as pd
from Bio import SeqIO
import sys
import os
from ast import literal_eval
import numpy as np
import pybedtools
from pybedtools import BedTool
from matplotlib.colors import LinearSegmentedColormap
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import gaussian_kde
from scipy.stats import pearsonr
from scipy.stats import spearmanr
import itertools
import re
import ast
import math

# --- Shared helpers: const.py lives in analyses/common ---
NB_DIR = os.getcwd()
sys.path.append(os.path.abspath(os.path.join(NB_DIR, '..', 'common')))
import const
from const import pos_active_ctrl_color, neg_active_ctrl_color, highlight_color, custom_cmap
from const import set_equal_plot_limits, plot_color_pallete, custom_cmap_bolder, FONT_SIZE_small
const.set_plot_style()

# --- Paths (EDIT): inputs read / outputs written by this notebook ---
CHONDROCYTES_DIR = '/home/labs/davidgo/Collaboration/humanMPRA/chondrocytes'  # per-library MPRAnalyze output
output_path = '.'  # where plots/tables are written

In [ ]:
# Count non-zero barcode entries in one post-filtering DNA chunk file (seq_325030)
file_path = os.path.join(CHONDROCYTES_DIR, 'L3a2', 'output', 'mpranalyze_comparative',
                         'dna_chunks_seq_325030_v2', 'dna_seq_325030_group1_v2.txt')

# Read the file and count non-zero numbers
with open(file_path, 'r') as f:
    content = f.read()

# Split by whitespace/newlines and convert to numbers, skipping non-numeric values
values = content.split()
numbers = []
for x in values:
    try:
        numbers.append(float(x))
    except ValueError:
        # Skip non-numeric values (like headers)
        pass

non_zero_count = sum(1 for num in numbers if num != 0)

print(f"Total numbers: {len(numbers)}")
print(f"Non-zero numbers: {non_zero_count}")

# Comparing number of barcodes before and after filtering for extreme outliers
#### Since we do not have the combined barcode count for all reps before and after 

In [ ]:
# Load ratio_wo_outliers tables for all requested libraries using only relevant columns
libraries = [
    "L1a1", "L1a2", "L1a3",
    "L2a1", "L2a2", "L2a3",
    "L3a1", "L3a2", "L3a3",
    "L4a1"
]

relevant_cols = [
    "oligo",
    "count_no_filter_rep1", "count_no_filter_rep2", "count_no_filter_rep3",
    "count_filtered_std2_rep1", "count_filtered_std2_rep2", "count_filtered_std2_rep3"
]

base_dir = CHONDROCYTES_DIR


def table_path_for_library(lib_name):
    return os.path.join(base_dir, lib_name, "output", "filter", "ratio_wo_outliers_std2.csv")


library_dfs = []
failed_libraries = []

for lib in libraries:
    lib_path = table_path_for_library(lib)
    try:
        lib_df = pd.read_csv(lib_path, usecols=relevant_cols)
        lib_df["library"] = lib
        library_dfs.append(lib_df)
        print(f"Loaded {lib}: {lib_df.shape[0]:,} rows")
    except Exception as exc:
        failed_libraries.append((lib, lib_path, str(exc)))
        print(f"FAILED {lib}: {exc}")

if not library_dfs:
    raise RuntimeError("No library table was loaded. Check path template and file availability.")

# Concatenate all loaded libraries (same selected columns + library tag)
ratio_wo_outliers_all = pd.concat(library_dfs, ignore_index=True)

count_cols_before = ["count_no_filter_rep1", "count_no_filter_rep2", "count_no_filter_rep3"]
count_cols_after = ["count_filtered_std2_rep1", "count_filtered_std2_rep2", "count_filtered_std2_rep3"]

# Ensure numeric before aggregation
for c in count_cols_before + count_cols_after:
    ratio_wo_outliers_all[c] = pd.to_numeric(ratio_wo_outliers_all[c], errors="coerce").fillna(0)

# Global totals across all loaded libraries
totals_by_rep = pd.DataFrame({
    "replicate": ["rep1", "rep2", "rep3"],
    "before_filtering": [
        ratio_wo_outliers_all["count_no_filter_rep1"].sum(),
        ratio_wo_outliers_all["count_no_filter_rep2"].sum(),
        ratio_wo_outliers_all["count_no_filter_rep3"].sum()
    ],
    "after_filtering_std2": [
        ratio_wo_outliers_all["count_filtered_std2_rep1"].sum(),
        ratio_wo_outliers_all["count_filtered_std2_rep2"].sum(),
        ratio_wo_outliers_all["count_filtered_std2_rep3"].sum()
    ]
})

totals_by_rep["removed"] = totals_by_rep["before_filtering"] - totals_by_rep["after_filtering_std2"]
totals_by_rep["pct_removed"] = np.where(
    totals_by_rep["before_filtering"] > 0,
    (totals_by_rep["removed"] / totals_by_rep["before_filtering"]) * 100,
    np.nan
)

print("\n" + "=" * 80)
print(f"Loaded libraries: {len(library_dfs)} / {len(libraries)}")
print(f"Concatenated shape: {ratio_wo_outliers_all.shape}")
print("=" * 80)

print("\nBarcode totals by replicate (all loaded libraries combined):")
print(totals_by_rep.to_string(index=False, float_format=lambda x: f"{x:,.2f}"))

# Optional: per-library totals
per_library_totals = ratio_wo_outliers_all.groupby("library", as_index=False).agg({
    "count_no_filter_rep1": "sum", "count_no_filter_rep2": "sum", "count_no_filter_rep3": "sum",
    "count_filtered_std2_rep1": "sum", "count_filtered_std2_rep2": "sum", "count_filtered_std2_rep3": "sum"
})

print("\nPer-library totals:")
print(per_library_totals.to_string(index=False, max_rows=20))

if failed_libraries:
    print("\nFailed libraries:")
    for lib, path, err in failed_libraries:
        print(f"  {lib}: {path}")
        print(f"    {err}")

In [ ]:
# Average removed barcodes per replicate, treating each library-replicate independently
replicate_level_rows = []

for _, row in per_library_totals.iterrows():
    lib = row["library"]
    for rep in [1, 2, 3]:
        before = row[f"count_no_filter_rep{rep}"]
        after = row[f"count_filtered_std2_rep{rep}"]
        removed = before - after
        pct_removed = (removed / before * 100) if before > 0 else np.nan

        replicate_level_rows.append({
            "library": lib,
            "replicate": f"rep{rep}",
            "before": before,
            "after": after,
            "removed": removed,
            "pct_removed": pct_removed,
        })

replicate_level_stats = pd.DataFrame(replicate_level_rows)

avg_removed_per_replicate = replicate_level_stats["removed"].mean()
avg_pct_removed_per_replicate = replicate_level_stats["pct_removed"].mean()

# Optional reference metric: pooled percent removed across all replicate entries
pooled_pct_removed = (
    replicate_level_stats["removed"].sum() / replicate_level_stats["before"].sum() * 100
)

print("=" * 80)
print("Per-replicate removal summary (library-replicates treated as independent):")
print("=" * 80)
print(f"Number of replicate entries: {len(replicate_level_stats)}")
print(f"Average removed barcodes per replicate: {avg_removed_per_replicate:,.2f}")
print(f"Average percent removed per replicate: {avg_pct_removed_per_replicate:.4f}%")
print(f"Pooled percent removed (reference): {pooled_pct_removed:.4f}%")

print("\nPer-replicate table (first 15 rows):")
print(replicate_level_stats.head(15).to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

In [ ]:
# Row-level loss per oligo per replicate (do NOT sum replicates)
row_level_loss_long_parts = []

for rep in [1, 2, 3]:
    before_col = f"count_no_filter_rep{rep}"
    after_col = f"count_filtered_std2_rep{rep}"

    part = ratio_wo_outliers_all[["library", "oligo", before_col, after_col]].copy()
    part = part.rename(columns={before_col: "before", after_col: "after"})
    part["replicate"] = f"rep{rep}"
    part["lost"] = part["before"] - part["after"]
    part["pct_lost"] = np.where(part["before"] > 0, (part["lost"] / part["before"]) * 100, np.nan)
    row_level_loss_long_parts.append(part)

row_level_loss_per_rep = pd.concat(row_level_loss_long_parts, ignore_index=True)

# Summary table: average row-level loss per replicate across all libraries combined
row_level_loss_summary = (
    row_level_loss_per_rep
    .groupby("replicate", as_index=False)
    .agg(
        n_rows=("oligo", "size"),
        avg_lost_per_row=("lost", "mean"),
        median_lost_per_row=("lost", "median"),
        avg_pct_lost_per_row=("pct_lost", "mean")
    )
)

overall_avg_lost_per_row_per_rep = row_level_loss_per_rep["lost"].mean()
overall_avg_pct_lost_per_row_per_rep = row_level_loss_per_rep["pct_lost"].mean()

print("=" * 80)
print("Row-level loss per oligo per replicate (all libraries combined)")
print("=" * 80)
print(f"Total row-replicate entries: {len(row_level_loss_per_rep):,}")
print(row_level_loss_summary.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))

print("\nOverall (across rep1/rep2/rep3 row-level entries):")
print(f"Average barcodes lost per oligo per replicate: {overall_avg_lost_per_row_per_rep:,.4f}")
print(f"Average percent lost per oligo per replicate: {overall_avg_pct_lost_per_row_per_rep:,.4f}%")